# UW-LYT-MS V2: Local Training, Evaluation & Profiling
Notebook huấn luyện, đánh giá và đo lường hiệu năng trên máy cục bộ (**Local Machine: Windows / Linux / macOS**).

### Mục tiêu & Điểm nổi bật:
1. **Mô hình siêu nhẹ `uwlytmsv2_3ch`:** Chỉ ~125k tham số (nhỏ hơn **250 lần** so với U-Net 31.4M trong bài báo gốc) nhưng kế thừa cấu trúc multiscale receptive field và xử lý tách biệt độ sáng/màu sắc (YCbCr).
2. **Tương thích phần cứng tự động:** Tự động phát hiện và chuyển đổi tối ưu giữa **NVIDIA GPU (CUDA)** và **CPU** mà không gây crash lỗi assertion.
3. **Tự động cấu hình dữ liệu:** Tự tìm kiếm EUVP dataset trong máy hoặc tạo dataset mini (Smoke Test) để kiểm tra toàn bộ luồng code ngay lập tức.
4. **Profiling & Trực quan hóa:** Đo lường trực tiếp Params, FLOPs, Latency (ms) và vẽ ảnh so sánh chất lượng ngay trong notebook.

In [ ]:
!pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu132
# ============================================================
# 1. THIẾT LẬP REPOSITORY & TỰ ĐỘNG PHÁT HIỆN PHẦN CỨNG
# ============================================================
import os
import sys
from pathlib import Path
import torch

# 1.1. Xác định thư mục gốc của repository
REPO_ROOT = Path.cwd()
if (REPO_ROOT / "src" / "uwir").exists():
    pass
elif (REPO_ROOT.parent / "src" / "uwir").exists():
    REPO_ROOT = REPO_ROOT.parent
else:
    curr = REPO_ROOT
    for _ in range(3):
        if (curr / "pyproject.toml").exists():
            REPO_ROOT = curr
            break
        curr = curr.parent

src_dir = REPO_ROOT / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
os.chdir(REPO_ROOT)

# 1.2. Phát hiện môi trường tính toán (CUDA GPU / CPU)
HAS_CUDA = torch.cuda.is_available()
DEVICE = "cuda" if HAS_CUDA else "cpu"
NUM_GPUS = torch.cuda.device_count() if HAS_CUDA else 0

print("=" * 65)
print(f"Thư mục Project  : {REPO_ROOT.resolve()}")
print(f"Trình thông dịch : {sys.executable}")
print(f"Phiên bản PyTorch: {torch.__version__}")
print(f"Thiết bị tính toán: {DEVICE.upper()} (Số lượng GPU khả dụng: {NUM_GPUS})")
print(torch.is_available())
if HAS_CUDA:
    for i in range(NUM_GPUS):
        print(f"  [+] GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("  [i] Chạy ở chế độ CPU (khuyên dùng mode 'smoke' khi test trên máy không có GPU)")
print("=" * 65)

Looking in indexes: https://download.pytorch.org/whl/cu132
Thư mục Project  : D:\eureka\underwater-image-enhancement
Trình thông dịch : c:\ProgramData\anaconda3\envs\hungpt19\python.exe
Phiên bản PyTorch: 2.13.0+cpu
Thiết bị tính toán: CPU (Số lượng GPU khả dụng: 0)
  [i] Chạy ở chế độ CPU (khuyên dùng mode 'smoke' khi test trên máy không có GPU)


In [ ]:
# ============================================================
# 2. KIỂM TRA MÔ HÌNH TRONG REGISTRY & ĐỐI CHIẾU SỐ THAM SỐ
# ============================================================
from uwir.models.registry import ALL_MODEL_NAMES, build_model

candidates = [
    ("UNet 5-channel (Bài báo gốc)", "unet_5ch"),
    ("UNet 3-channel (RGB Baseline)", "unet_3ch"),
    ("UW-LYT-MS V2 (Khuyến nghị - Siêu nhẹ)", "uwlytmsv2_3ch"),
    ("UW-LYT-MS V1 (Multiscale)", "uwlytms_3ch"),
    ("UW-LYT V2 (Asymmetric Cb/Cr)", "uwlytv2_3ch"),
    ("UW-LYT V1 (Flat Residual)", "uwlyt_3ch"),
    ("UW-LYT-Tiny (Cực nhẹ ~12k)", "uwlyttiny_3ch"),
]

print(f"{'Mô hình':<40} {'Tên định danh':<18} {'Số tham số':>12} {'Tỉ lệ giảm':>12}")
print("-" * 86)
base_params = sum(p.numel() for p in build_model("unet_5ch").parameters())
for desc, name in candidates:
    m = build_model(name)
    n_params = sum(p.numel() for p in m.parameters())
    ratio = f"{base_params / n_params:.1f}x"
    print(f"{desc:<40} {name:<18} {n_params:>12,} {ratio:>12}")

In [ ]:
# ============================================================
# 3. TỰ ĐỘNG TÌM KIẾM DATASET EUVP HOẶC TẠO DATASET MINI (TEST)
# ============================================================
# Nếu bạn để bộ dữ liệu EUVP ở thư mục riêng, hãy điền vào biến dưới đây:
CUSTOM_EUVP_DIR = None  # Ví dụ: r"D:\datasets\EUVP" hoặc r"C:\data\EUVP"

candidate_paths = [
    Path(CUSTOM_EUVP_DIR) if CUSTOM_EUVP_DIR else None,
    REPO_ROOT / "datasets" / "EUVP",
    REPO_ROOT.parent / "datasets" / "EUVP",
    Path("D:/datasets/EUVP"),
    Path("C:/datasets/EUVP"),
]

EUVP_ROOT = next((p for p in candidate_paths if p and p.exists() and (p / "test_samples" / "Inp").exists()), None)

if EUVP_ROOT is None:
    print("[THÔNG BÁO] Chưa phát hiện bộ dữ liệu EUVP đầy đủ tại các đường dẫn mặc định.")
    EUVP_ROOT = REPO_ROOT / "datasets" / "EUVP_mini_smoke"
    print(f"--> Tự động thiết lập thư mục dữ liệu mẫu (Dummy/Mini) tại: {EUVP_ROOT}")
    
    # Tạo cấu trúc thư mục EUVP chuẩn
    for split in ["underwater_imagenet", "underwater_dark", "underwater_scenes"]:
        (EUVP_ROOT / "Paired" / split / "trainA").mkdir(parents=True, exist_ok=True)
        (EUVP_ROOT / "Paired" / split / "trainB").mkdir(parents=True, exist_ok=True)
    (EUVP_ROOT / "test_samples" / "Inp").mkdir(parents=True, exist_ok=True)
    (EUVP_ROOT / "test_samples" / "GTr").mkdir(parents=True, exist_ok=True)
    
    # Tạo vài cặp ảnh mẫu ngẫu nhiên nếu thư mục đang trống
    from PIL import Image
    import numpy as np
    existing = list((EUVP_ROOT / "test_samples" / "Inp").glob("*.jpg"))
    if not existing:
        np.random.seed(42)
        for i in range(8):
            img_in = Image.fromarray((np.random.rand(256, 256, 3) * 255).astype(np.uint8))
            img_gt = Image.fromarray((np.random.rand(256, 256, 3) * 255).astype(np.uint8))
            subset = ["underwater_imagenet", "underwater_dark", "underwater_scenes"][i % 3]
            img_in.save(EUVP_ROOT / "Paired" / subset / "trainA" / f"img_{i}.jpg")
            img_gt.save(EUVP_ROOT / "Paired" / subset / "trainB" / f"img_{i}.jpg")
            if i < 4:
                img_in.save(EUVP_ROOT / "test_samples" / "Inp" / f"test_{i}.jpg")
                img_gt.save(EUVP_ROOT / "test_samples" / "GTr" / f"test_{i}.jpg")
        print("    [OK] Đã tạo xong 8 ảnh train và 4 ảnh test mẫu để chạy kiểm thử luồng code!")
else:
    print(f"[OK] Đã tìm thấy bộ dữ liệu EUVP chính thức tại: {EUVP_ROOT}")

In [ ]:
# ============================================================
# 4. CẤU HÌNH THAM SỐ THỰC NGHIỆM (LOCAL HYPERPARAMETERS)
# ============================================================
MODEL_VARIANT = "uwlytmsv2_3ch"   # Lựa chọn: 'uwlytmsv2_3ch', 'uwlytms_3ch', 'uwlytv2_3ch', 'unet_5ch'

# Đặt SMOKE = True nếu bạn muốn kiểm tra chạy thử nghiệm nhanh (1 epoch, 1 seed)
# Đặt SMOKE = False nếu bạn muốn huấn luyện đầy đủ (50 epochs, 3 seeds) với GPU
SMOKE = True

if SMOKE:
    EPOCHS = 1
    RUNS = 1
    SEEDS = "0"
    TAG = "smoke"
    BATCH_SIZE = 4 if not HAS_CUDA else 16
else:
    EPOCHS = 50
    RUNS = 3
    SEEDS = "0 1 2"
    TAG = "full"
    BATCH_SIZE = 16 if HAS_CUDA else 4

CROP_SIZE = 256
# Trên Windows hoặc khi chạy CPU, threads=0 giúp tránh lỗi phân chia tiến trình bộ nhớ
THREADS = 2 if (HAS_CUDA and os.name != "nt") else 0

CHECKPOINTS = REPO_ROOT / f"checkpoints_{MODEL_VARIANT}_{TAG}"
RESULTS = REPO_ROOT / f"results_{MODEL_VARIANT}_{TAG}"

print("CẤU HÌNH ĐƯỢC CHỌN:")
print(f"  - Mô hình      : {MODEL_VARIANT}")
print(f"  - Chế độ chạy  : {TAG.upper()} ({EPOCHS} epochs, {RUNS} runs, seeds: {SEEDS})")
print(f"  - Batch size   : {BATCH_SIZE} | Crop size: {CROP_SIZE} | Worker threads: {THREADS}")
print(f"  - Checkpoints  : {CHECKPOINTS}")
print(f"  - Kết quả xuất : {RESULTS}")

In [ ]:
# ============================================================
# 5. THỰC THI HUẤN LUYỆN (TRAINING EXECUTION)
# ============================================================
import subprocess

train_cmd = [
    sys.executable, "-m", "scripts.experiments.ablation_euvp",
    "--data_train_euvp", str(EUVP_ROOT),
    "--checkpoint_dir", str(CHECKPOINTS),
    "--val_folder", str(RESULTS),
    "--variants", MODEL_VARIANT,
    "--nEpochs", str(EPOCHS),
    "--batchSize", str(BATCH_SIZE),
    "--cropSize", str(CROP_SIZE),
    "--lr", "1e-4",
    "--weight_decay", "1e-5",
    "--L1_weight", "1.0",
    "--perceptual_weight", "1.0",
    "--SSIM_weight", "0.0",
    "--scheduler_step", "30",
    "--scheduler_gamma", "0.5",
    "--early_stop_patience", "20",
    "--num_runs", str(RUNS),
    "--seeds", *SEEDS.split(),
    "--threads", str(THREADS),
    "--num_gpus", str(max(1, NUM_GPUS)),
]
if not HAS_CUDA:
    train_cmd.append("--no_gpu")

print("Bắt đầu huấn luyện...")
print("Lệnh thực thi:", " ".join(train_cmd))
print("-" * 65)

# Chạy lệnh và truyền log trực tiếp vào output notebook
process = subprocess.Popen(
    train_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in iter(process.stdout.readline, ""):
    print(line, end="")
process.stdout.close()
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Lỗi trong quá trình huấn luyện (Mã lỗi: {return_code})")

print("\n[HOÀN TẤT HUẤN LUYỆN THÀNH CÔNG]")

In [ ]:
# ============================================================
# 6. ĐÁNH GIÁ CHẤT LƯỢNG MÔ HÌNH (PSNR, SSIM, CIEDE2000, UIQM)
# ============================================================
eval_cmd = [
    sys.executable, "-m", "uwir.cli.evaluate",
    "--eval_benchmark", "euvp",
    "--data_train_euvp", str(EUVP_ROOT),
    "--checkpoint_dir", str(CHECKPOINTS),
    "--val_folder", str(RESULTS / "evaluation"),
    "--batchSize", str(BATCH_SIZE),
    "--cropSize", str(CROP_SIZE),
    "--threads", str(THREADS),
    "--gpu_mode", "True" if HAS_CUDA else "False",
    "--native_eval", "True",
]

print("Bắt đầu đánh giá mô hình trên tập kiểm thử...")
print("-" * 65)
eval_proc = subprocess.run(eval_cmd, capture_output=True, text=True)
print(eval_proc.stdout)
if eval_proc.returncode != 0:
    print("Lỗi stderr:", eval_proc.stderr)
else:
    print("[HOÀN TẤT ĐÁNH GIÁ]")

In [ ]:
# ============================================================
# 7. BENCHMARK PROFILING (ĐO FLOPs, LATENCY & BỘ NHỚ)
# ============================================================
import pandas as pd

profile_dir = RESULTS / "profile"
profile_dir.mkdir(parents=True, exist_ok=True)

# So sánh mô hình đang chọn với U-Net 5ch bài báo và baseline RGB
models_to_profile = list({MODEL_VARIANT, "unet_5ch", "unet_3ch"})

profile_cmd = [
    sys.executable, "-m", "uwir.cli.profile",
    *models_to_profile,
    "--device", DEVICE,
    "--no-pretrained",
    "--img-size", "256",
    "--runs", "5",
    "--output-dir", str(profile_dir),
]

print("Đang tiến hành đo lường hiệu năng phần cứng...")
subprocess.run(profile_cmd, check=True)

# Đọc và hiển thị bảng so sánh CSV vừa sinh ra
csv_files = sorted(profile_dir.glob("*.csv"))
if csv_files:
    latest_csv = csv_files[-1]
    df = pd.read_csv(latest_csv)
    print("\n" + "=" * 65)
    print("  BẢNG TỔNG HỢP HIỆU NĂNG MÔ HÌNH TRÊN MÁY BẠN")
    print("=" * 65)
    cols = ["model", "in_channels", "params_M", "flops_G", "peak_memory_MiB", "model_time_ms", "combined_time_ms"]
    print(df[cols].to_markdown(index=False))
    try:
        display(df[cols])
    except NameError:
        pass

In [ ]:
# ============================================================
# 8. TRỰC QUAN HÓA KẾT QUẢ ĐỊNH TÍNH (INPUT vs RESTORED vs GT)
# ============================================================
import matplotlib.pyplot as plt
from PIL import Image

test_inp_dir = EUVP_ROOT / "test_samples" / "Inp"
test_gt_dir = EUVP_ROOT / "test_samples" / "GTr"
eval_out_dir = RESULTS / "evaluation"

test_imgs = sorted(list(test_inp_dir.glob("*.jpg")) + list(test_inp_dir.glob("*.png")))[:3]

if test_imgs:
    fig, axes = plt.subplots(len(test_imgs), 3, figsize=(12, 4 * len(test_imgs)))
    if len(test_imgs) == 1:
        axes = [axes]
        
    for idx, inp_path in enumerate(test_imgs):
        stem = inp_path.stem
        gt_path = test_gt_dir / inp_path.name
        
        # Tìm ảnh phục hồi nếu có trong thư mục evaluation
        pred_candidates = list(eval_out_dir.glob(f"**/{stem}*"))
        pred_path = pred_candidates[0] if pred_candidates else None
        
        inp_img = Image.open(inp_path)
        gt_img = Image.open(gt_path) if gt_path.exists() else None
        pred_img = Image.open(pred_path) if pred_path and pred_path.exists() else None
        
        axes[idx][0].imshow(inp_img)
        axes[idx][0].set_title(f"Input (Degraded)\n{inp_path.name}", fontsize=11)
        axes[idx][0].axis("off")
        
        if pred_img:
            axes[idx][1].imshow(pred_img)
            axes[idx][1].set_title(f"Restored ({MODEL_VARIANT})", fontsize=11)
        else:
            axes[idx][1].text(0.5, 0.5, f"Restored Model\n({MODEL_VARIANT})", ha="center", va="center", color="blue")
        axes[idx][1].axis("off")
        
        if gt_img:
            axes[idx][2].imshow(gt_img)
            axes[idx][2].set_title("Ground Truth (Reference)", fontsize=11)
        else:
            axes[idx][2].axis("off")
        axes[idx][2].axis("off")
        
    plt.suptitle("So sánh chất lượng phục hồi ảnh dưới nước", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("Không có ảnh kiểm thử trong thư mục test_samples để hiển thị.")